# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to programmatically load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) dataset using the `mlcroissant` Python library.

### Dataset Source

The data is described using the [Croissant standard](https://mlcommons.org/croissant/) with a schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

This dataset contains tabular clinical and molecular information for 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print dataset-level information
metadata = dataset.metadata  # <-- NOT a dict, don't use [..]
print(f"{metadata.name}: {metadata.description}")
print(f"\nIdentifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review the dataset's available record sets and fields, referencing each by its `@id` as specified by the Croissant schema.

In [ ]:
# List available record sets and their fields with @id
record_sets = list(dataset.record_sets())
print(f"Number of record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

# Pick the main clinical record set @id for downstream examples
# If only one, use it. Change this id variable if you want to focus on a different set.
main_record_set_id = record_sets[0].id if record_sets else None
print(f"Main record set selected for analysis: {main_record_set_id}")

## 3. Data Extraction

Load all records from the main record set into a Pandas DataFrame, referencing the record set and field `@id`s obtained above.

You may repeat the below for additional record sets if desired.

In [ ]:
# Extract all data from each record set to DataFrames, using their @id
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"DataFrame for record set '{rs.name}': {df.shape[0]} rows, {df.shape[1]} columns.")

# Show the available column (field) @id values for the main record set
main_df = dataframes[main_record_set_id]
print(f"\nColumns (field @id) in main DataFrame: {list(main_df.columns)}")

# Show first few records
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's analyze one of the numeric fields (for example, age at diagnosis) and perform basic cleaning and transformation. In this example, we will demonstrate filtering, normalization, and simple aggregation/grouping.

All field references are by `@id`.

In [ ]:
# Choose a numeric field (@id) – adapt as appropriate for your dataset

# List of field @ids that look like numeric fields (by simple pattern matching)
numeric_field_id = None
for col in main_df.columns:
    if any(keyword in col.lower() for keyword in ['age', 'interval', 'years', 'count', 'number']):
        numeric_field_id = col
        print(f"Using numeric field: {numeric_field_id}")
        break

# Default fallback if a numeric field couldn't be found automatically
if numeric_field_id is None:
    numeric_field_id = main_df.columns[0]
    print(f"Warning: Could not determine numeric field, defaulting to: {numeric_field_id}")

threshold = 50  # Set a threshold appropriate for age/interval
# Ensure column is numeric
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-scoring)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a categorical group field for aggregation (try to find automatically)
group_field_id = None
for col in main_df.columns:
    if any(keyword in col.lower() for keyword in ['sex', 'gender', 'msi', 'anatomical', 'location', 'site', 'type', 'status', 'group']):
        group_field_id = col
        print(f"Using group field: {group_field_id}")
        break

# Mean of numeric field by group
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization

Let's visualize the distribution of our selected numeric field and compare across a grouping variable, if available.

All axes and legends show underlying field `@id`s for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=20, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

# Boxplot by group, if applicable
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load and explore the FAIR^2 dataset, referenced record sets and fields by their Croissant `@id`, and performed basic analysis and visualization. This approach ensures reproducibility and adherence to dataset metadata standards.

- Record sets, fields, and columns were referenced by their respective `@id` fields, as shown.
- Data was filtered, normalized, and visualized based on numeric and categorical fields, also referenced by `@id`.

For further analysis, consult the dataset's Croissant schema for more detailed field descriptions or explore additional record sets if available.